In [22]:
# Imports and config

import os, torch, pandas as pd, numpy as np
from PIL import Image
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import tempfile, shutil

# ── Config ─────────────────────────────────────────────────────
CSV_PATH     = '/kaggle/input/datasets/pankajdeopa/vr-processed-data/master_labels.csv'
IMG_SAVE_DIR = '/kaggle/working/preprocessed_dataset/images/'
OUT_CSV      = '/kaggle/working/preprocessed_dataset/master_labels_preprocessed.csv'
IMG_SIZE     = 224
NUM_WORKERS  = 4

os.makedirs(IMG_SAVE_DIR, exist_ok=True)
print("Config ready")

Config ready


In [23]:
# Worker function

def process_row(args):
    idx, row = args
    try:
        img = Image.open(row['image_path']).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        save_path = IMG_SAVE_DIR + f"{idx}.jpg"
        img.save(save_path, "JPEG", quality=95)

        new_row = row.copy()
        new_row['image_path'] = save_path
        return new_row

    except Exception as e:
        print(f"ERROR at {row['image_path']}: {e}")
        return None

In [24]:
# Preprocessing

df   = pd.read_csv(CSV_PATH)
rows = list(df.iterrows())

print(f"Total images : {len(rows)}")
print(f"Workers      : {NUM_WORKERS}")
print(f"Estimated size: ~{len(rows) * 20 / 1024:.1f} MB")

with Pool(NUM_WORKERS) as p:
    results = list(tqdm(
        p.imap(process_row, rows),
        total=len(rows)
    ))

# Filter failed rows
results = [r for r in results if r is not None]
print(f"\nSuccessfully processed: {len(results)}/{len(rows)}")

Total images : 167915
Workers      : 4
Estimated size: ~3279.6 MB


100%|██████████| 167915/167915 [14:28<00:00, 193.37it/s]


Successfully processed: 167915/167915


In [ ]:
# save as CSV

new_df = pd.DataFrame(results)
new_df.to_csv(OUT_CSV, index=False)
print(f"Saved CSV: {OUT_CSV}")
print(f"Sample path: {new_df['image_path'].iloc[0]}")

# Check disk usage
import subprocess
result = subprocess.run(['du', '-sh', '/kaggle/working/preprocessed_dataset/'],
                       capture_output=True, text=True)
print(f"Disk usage: {result.stdout}")

Saved CSV: /kaggle/working/preprocessed_dataset/master_labels_preprocessed.csv
Sample path: /kaggle/working/preprocessed_dataset/images/0.jpg
Disk usage: 4.5G	/kaggle/working/preprocessed_dataset/



In [ ]:
# Upload dataset to kaggle

import json, subprocess

metadata = {
    "title"   : "vr-resnet50-preprocessed-images",
    "id"      : "pankajdeopa/vr-resnet50-preprocessed-images",
    "licenses": [{"name": "CC0-1.0"}]
}
with open('/kaggle/working/preprocessed_dataset/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p',
     '/kaggle/working/preprocessed_dataset/', '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Starting upload for file master_labels_preprocessed.csv
Upload successful: master_labels_preprocessed.csv (11MB)
Starting upload for file images.zip
Upload successful: images.zip (3GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopa/vr-resnet50-preprocessed-images


  0%|          | 0.00/11.2M [00:00<?, ?B/s]
 17%|█▋        | 1.92M/11.2M [00:00<00:00, 20.0MB/s]
100%|██████████| 11.2M/11.2M [00:00<00:00, 17.1MB/s]

  0%|          | 0.00/3.42G [00:00<?, ?B/s]
  0%|          | 8.52M/3.42G [00:00<01:02, 58.9MB/s]
  1%|          | 27.6M/3.42G [00:00<00:29, 124MB/s] 
  1%|          | 40.6M/3.42G [00:00<00:37, 95.9MB/s]
  1%|▏         | 50.8M/3.42G [00:00<00:47, 76.3MB/s]
  2%|▏         | 65.8M/3.42G [00:00<00:41, 86.1MB/s]
  2%|▏         | 81.8M/3.42G [00:00<00:36, 98.7MB/s]
  3%|▎         | 91.9M/3.42G [00:01<00:38, 92.0MB/s]
  3%|▎         | 106M/3.42G [00:01<00:39, 90.7MB/s] 
  3%|▎         | 122M/3.42G [00:01<00:36, 97.3MB/s]
  